# Pipeline completo — Detector de Plagio

Este notebook muestra cómo usar el pipeline de principio a fin:

```
archivos .py  →  ast_tokenizer  →  winnowing  →  features  →  build_pairs  →  model
```

> **Prerequisito:** la carpeta `pipeline/` debe estar en el mismo directorio que este notebook.

## 0. Imports

In [4]:
import sys
sys.path.insert(0, '.')   # asegura que Python encuentre la carpeta pipeline/

from pipeline.ast_tokenizer import tokenize_file, tokenize_source
from pipeline.winnowing      import winnow, winnowing_similarity, fingerprint_jaccard
from pipeline.features       import compute_features, FEATURE_COLUMNS
from pipeline.build_pairs    import build_pairs_csv

import pandas as pd

---
## 1. Tokenizar un archivo individual

El primer paso del pipeline: convertir un `.py` en una secuencia de tokens normalizados.

In [ ]:
resultado = tokenize_file("data/fibonacci/snippets/snip_01.py", masking="medium")

print("Tokens     :", resultado["tokens"])
print("Profundidad:", resultado["max_depth"])
print("Error      :", resultado["error"])     # None si todo fue bien

Tokens     : ['Module', 'FunctionDef', 'VAR', 'arguments', 'VAR', 'Assign', 'VAR', 'Store', 'List', 'LIT', 'LIT', 'Load', 'For', 'VAR', 'Store', 'Call', 'VAR', 'Load', 'LIT', 'VAR', 'Load', 'VAR', 'Load', 'LIT', 'VAR', 'Load', 'Expr', 'Call', 'Attribute', 'VAR', 'Load', 'VAR', 'Load', 'Load', 'BinOp', 'Subscript', 'VAR', 'Load', 'UnaryOp', 'USub', 'LIT', 'Load', 'Add', 'Subscript', 'VAR', 'Load', 'UnaryOp', 'USub', 'LIT', 'Load', 'Attribute', 'VAR', 'Load', 'VAR', 'Load', 'Load', 'BinOp', 'Subscript', 'VAR', 'Load', 'UnaryOp', 'USub', 'LIT', 'Load', 'Add', 'Subscript', 'VAR', 'Load', 'UnaryOp', 'USub', 'LIT', 'Load', 'Return', 'Subscript', 'VAR', 'Load', 'Slice', 'VAR', 'Load', 'Load']
Profundidad: 9
Error      : None


---
## 2. Generar huellas digitales (Winnowing)

A partir de los tokens, Winnowing produce un `frozenset` de hashes que representa la huella del archivo.

In [12]:
tokens = resultado["tokens"]

# k=23 y w=4 son los valores por defecto de Dolos
# Para archivos cortos usa k más pequeño (ej. k=5)
fingerprints = winnow(tokens, k=5, w=4)

print(f"Número de huellas : {len(fingerprints)}")
print(f"Primeras 5        : {sorted(fingerprints)[:5]}")

Número de huellas : 21
Primeras 5        : [130764251349465257, 225615047730256977, 546784712304640041, 1299496419633592567, 1615824289367989767]


---
## 3. Comparar dos archivos

Calcula las 5 características para un par.

In [13]:
# ── Archivo A ──────────────────────────────────────────────
src_a = """
def reverse_string(s):
    result = ''
    for char in s:
        result = char + result
    return result
"""

# ── Archivo B (mismo problema, distinta implementación) ────
src_b = """
def flip_str(text):
    out = ''
    i = len(text) - 1
    while i >= 0:
        out += text[i]
        i -= 1
    return out
"""

K = 5   # usa k=23 con archivos reales del dataset

ra = tokenize_source(src_a, masking="medium")
rb = tokenize_source(src_b, masking="medium")

fps_a = winnow(ra["tokens"], k=K, w=4)
fps_b = winnow(rb["tokens"], k=K, w=4)

features = compute_features(
    ra["tokens"], ra["max_depth"], fps_a,
    rb["tokens"], rb["max_depth"], fps_b,
    k=K,
)

for nombre, valor in features.items():
    print(f"{nombre:30s}: {valor:.3f}")

winnowing_similarity          : 0.250
shared_fragment_coverage      : 0.267
token_overlap_ratio           : 0.085
ast_depth_difference          : 0.000
fingerprint_jaccard           : 0.095


---
## 4. Generar el CSV completo desde el dataset

Este es el paso principal: procesa todos los archivos del dataset de Kaggle y genera `pairs_features.csv`.

In [15]:
build_pairs_csv(
    data_dir = "data",
    output   = "pairs_features.csv",
    k        = 23,
    w        = 4,
    masking  = "medium",
    verbose  = True,
)

Problemas encontrados : 5
Archivos .py totales  : 100
  [  5/5] reverse_string — 20 archivos
Pares positivos (label=1): 950
Pares negativos (label=0): 950

CSV guardado en : pairs_features.csv
Total de filas  : 1900


WindowsPath('pairs_features.csv')

---
## 5. Inspeccionar el CSV generado

In [ ]:
df = pd.read_csv("pairs_features.csv")

print(f"Filas totales : {len(df)}")
print(f"Positivos     : {(df.label == 1).sum()}")
print(f"Negativos     : {(df.label == 0).sum()}")
print()
df.head(10)

In [ ]:
# Estadísticas por clase: los positivos deben tener valores más altos
# en las primeras 4 columnas y más bajo en ast_depth_difference
df.groupby("label")[FEATURE_COLUMNS].mean().round(3)

In [ ]:
# Distribución de cada característica por clase
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(FEATURE_COLUMNS), figsize=(18, 4))

for ax, col in zip(axes, FEATURE_COLUMNS):
    for label, color, name in [(1, "steelblue", "Plagio"), (0, "salmon", "No plagio")]:
        df[df.label == label][col].hist(ax=ax, alpha=0.6, bins=20,
                                        color=color, label=name)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## 6. Entrenar el modelo

Con el CSV listo, abre `model.ipynb` y ejecútalo.
Detecta `pairs_features.csv` automáticamente y entrena con los datos reales.

O bien ejecuta las celdas del modelo directamente aquí:

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
LABEL_COLUMN = "label"
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

x = df[FEATURE_COLUMNS].values.astype("float32")
y = df[LABEL_COLUMN].values.astype("float32")

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)

model = keras.Sequential([
    layers.Input(shape=(len(FEATURE_COLUMNS),)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1,  activation="sigmoid"),
])
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)

model.fit(x_train, y_train, epochs=30, batch_size=32,
          validation_split=0.2, verbose=1)

results = model.evaluate(x_test, y_test, return_dict=True)
p, r = results["precision"], results["recall"]
results["f1"] = 2*p*r/(p+r) if p+r > 0 else 0

print("\n── Resultados en test ──")
for k_name, v in results.items():
    print(f"  {k_name:12s}: {v:.3f}")

---
## 7. Usar el modelo para predecir un par nuevo

In [ ]:
def predict_pair(path_a: str, path_b: str, k: int = 23, w: int = 4,
                 masking: str = "medium") -> dict:
    """Dado dos archivos .py, devuelve la probabilidad de plagio."""
    ra = tokenize_file(path_a, masking=masking)
    rb = tokenize_file(path_b, masking=masking)

    fps_a = winnow(ra["tokens"], k=k, w=w)
    fps_b = winnow(rb["tokens"], k=k, w=w)

    feats = compute_features(
        ra["tokens"], ra["max_depth"], fps_a,
        rb["tokens"], rb["max_depth"], fps_b,
        k=k,
    )

    x_new = scaler.transform([[feats[c] for c in FEATURE_COLUMNS]])
    prob  = float(model.predict(x_new, verbose=0)[0][0])

    return {
        "probabilidad_plagio": round(prob, 4),
        "veredicto"          : "PLAGIO" if prob >= 0.5 else "NO PLAGIO",
        "features"           : {k: round(v, 4) for k, v in feats.items()},
    }


# Ejemplo de uso:
resultado = predict_pair("ruta/archivo_a.py", "ruta/archivo_b.py")
print(f"Veredicto           : {resultado['veredicto']}")
print(f"Probabilidad plagio : {resultado['probabilidad_plagio']}")
print("Features:")
for nombre, valor in resultado["features"].items():
    print(f"  {nombre:30s}: {valor}")